# 03 - Position Sizing Sanity Checks

This notebook turns the probability-gap framework into simple sizing diagnostics. The sizing rules here are intentionally conservative and transparent: they are not a complete live trading system.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.append(str(SRC))

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 180)

from biotech_catalyst.implied_probability import market_implied_probability, scenario_prices
from biotech_catalyst.position_sizing import edge_scaled_weight, expected_return_from_scenarios, kelly_fraction_binary


In [ ]:
returns = pd.read_csv(ROOT / "data" / "biotech_event_returns.csv")
case = pd.read_csv(ROOT / "data" / "cytokinetics_case_study.csv").iloc[0]

display(returns[["ticker", "outcome", "abnormal_return_5d"]].head())


In [ ]:
current_price = case["current_price"]
approval_price = case["approval_price"]
failure_price = case["failure_price"]
implied_probability = market_implied_probability(current_price, approval_price, failure_price)

probability_grid = np.linspace(implied_probability - 0.10, 0.95, 10)
rows = []
for p_star in probability_grid:
    expected_return = expected_return_from_scenarios(current_price, approval_price, failure_price, p_star)
    weight = edge_scaled_weight(p_star, implied_probability, neutral_band=0.05, max_weight=0.10)
    rows.append(
        {
            "model_probability": p_star,
            "probability_edge": p_star - implied_probability,
            "expected_return": expected_return,
            "edge_scaled_weight": weight,
        }
    )

sizing_table = pd.DataFrame(rows)
display(sizing_table.style.format({
    "model_probability": "{:.1%}",
    "probability_edge": "{:.1%}",
    "expected_return": "{:.1%}",
    "edge_scaled_weight": "{:.1%}",
}))


In [ ]:
fig, ax1 = plt.subplots(figsize=(8, 4))
ax1.plot(sizing_table["model_probability"], sizing_table["expected_return"], marker="o", label="expected return")
ax1.axhline(0, color="black", linewidth=1)
ax1.set_xlabel("Model approval probability")
ax1.set_ylabel("Expected return")

ax2 = ax1.twinx()
ax2.plot(sizing_table["model_probability"], sizing_table["edge_scaled_weight"], marker="s", color="tab:orange", label="position weight")
ax2.set_ylabel("Position weight")

ax1.set_title("Expected return and capped sizing by model probability")
plt.show()


In [ ]:
approval_return = approval_price / current_price - 1
failure_return = failure_price / current_price - 1
kelly_fraction = kelly_fraction_binary(case["model_probability"], approval_return, failure_return)

print("approval return:", round(approval_return, 3))
print("failure return:", round(failure_return, 3))
print("rough binary Kelly fraction:", round(kelly_fraction, 3))
print("Kelly is treated as a diagnostic here, not as a sizing rule.")
